In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))

from src.data_loader import read_csv_from_blob
from src.dataset import assemble_dataset
from src.train import train_per_horizon_models, predict_next_day

STORAGE_ACCOUNT = "sagreekdamdevweu"
GATE_CLOSURE_HOUR = 12
TARGET_DAY_ATHENS = pd.Timestamp("2026-05-14", tz="Europe/Athens")

print(f"Target prediction day (Athens): {TARGET_DAY_ATHENS.date()}")

Target prediction day (Athens): 2026-05-14


In [4]:
print("Loading processed/ data from blob...")

dam = read_csv_from_blob(STORAGE_ACCOUNT, "processed", "dam_prices/full.csv")
load = read_csv_from_blob(STORAGE_ACCOUNT, "processed", "load_forecast/full.csv")
renewable = read_csv_from_blob(STORAGE_ACCOUNT, "processed", "renewable_forecast/full.csv")

print(f"DAM:        {len(dam)} rows, {dam.index.min()} → {dam.index.max()}")
print(f"Load:       {len(load)} rows, {load.index.min()} → {load.index.max()}")
print(f"Renewable:  {len(renewable)} rows, {renewable.index.min()} → {renewable.index.max()}")

Loading processed/ data from blob...
DAM:        29472 rows, 2022-12-31 22:00:00+00:00 → 2026-05-12 21:00:00+00:00
Load:       29519 rows, 2022-12-31 22:00:00+00:00 → 2026-05-14 20:00:00+00:00
Renewable:  29519 rows, 2022-12-31 22:00:00+00:00 → 2026-05-14 20:00:00+00:00


In [5]:
load

,load_forecast_mw
2022-12-31 22:00:00+00:00,4269.0
2022-12-31 23:00:00+00:00,4008.0
2023-01-01 00:00:00+00:00,3888.0
2023-01-01 01:00:00+00:00,3760.0
2023-01-01 02:00:00+00:00,3703.0
...,...
2026-05-14 16:00:00+00:00,5452.5
2026-05-14 17:00:00+00:00,5735.0
2026-05-14 18:00:00+00:00,5770.0
2026-05-14 19:00:00+00:00,5272.5


In [6]:
prices_train, exog_train = assemble_dataset(dam, load, renewable, join="inner")
print(f"Training data (inner join): {len(prices_train)} rows")
print(f"Range: {prices_train.index.min()} → {prices_train.index.max()}")

Training data (inner join): 29472 rows
Range: 2022-12-31 22:00:00+00:00 → 2026-05-12 21:00:00+00:00


In [7]:
exog_train

,load_forecast_mw,solar,wind_onshore
2022-12-31 22:00:00+00:00,4269.0,0.0,495.0
2022-12-31 23:00:00+00:00,4008.0,0.0,518.0
2023-01-01 00:00:00+00:00,3888.0,0.0,560.0
2023-01-01 01:00:00+00:00,3760.0,0.0,603.0
2023-01-01 02:00:00+00:00,3703.0,0.0,649.0
...,...,...,...
2026-05-12 17:00:00+00:00,5787.5,45.0,2975.0
2026-05-12 18:00:00+00:00,5847.5,0.0,2872.5
2026-05-12 19:00:00+00:00,5342.5,0.0,2797.5
2026-05-12 20:00:00+00:00,4840.0,0.0,2710.0


In [8]:
lgbm_params = {
    "n_estimators": 200,
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_child_samples": 20,
    "random_state": 42,
    "verbose": -1,
}

result = train_per_horizon_models(
    prices_train,
    exog=exog_train,
    gate_closure_hour=GATE_CLOSURE_HOUR,
    horizons=tuple(range(0, 24)),
    same_hour_lag_days=(1, 2, 7),
    context_window=1,
    test_days=30,
    lgbm_params=lgbm_params,
)

In [9]:
result.X_train

,ft_price_minus_0h,ft_price_minus_1h,ft_price_minus_2h,ft_price_minus_3h,ft_mean_24h,ft_std_24h,ft_max_24h,ft_min_24h,ft_mean_168h,ft_std_168h,...,target_is_holiday,target_hour_sin,target_hour_cos,target_dow_sin,target_dow_cos,target_load_forecast_mw,target_solar,target_wind_onshore,target_net_load_mw,horizon
0,81.9500,84.9000,168.5000,177.2500,200.246000,54.846832,268.1900,81.9500,200.246000,54.846832,...,0,0.000000,1.000000,0.000000,1.000000,3562.0,0.0,527.0,3035.0,0
1,188.8900,187.9100,188.8900,196.2300,250.290417,68.863390,363.1500,179.7300,231.042564,67.726661,...,0,0.000000,1.000000,0.781831,0.623490,4107.0,0.0,151.0,3956.0,0
2,174.7600,175.0000,180.2000,191.5800,279.277083,71.240242,363.7700,172.7800,249.417619,72.467718,...,0,0.000000,1.000000,0.974928,-0.222521,4199.0,0.0,240.0,3959.0,0
3,172.0000,169.4100,173.5000,183.8300,231.874583,75.783884,369.1700,130.0000,244.578161,73.376974,...,0,0.000000,1.000000,0.433884,-0.900969,4267.0,0.0,883.0,3384.0,0
4,165.1000,165.0000,184.3800,192.9400,224.231250,82.200167,362.3100,100.0000,240.178829,75.452277,...,1,0.000000,1.000000,-0.433884,-0.900969,4284.0,0.0,575.0,3709.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28699,-3.4400,-4.9975,-3.0100,-0.2725,84.846771,67.366720,167.2625,-4.9975,99.355074,63.739238,...,0,-0.258819,0.965926,0.974928,-0.222521,4197.5,0.0,550.0,3647.5,23
28700,-6.4625,-8.4775,-6.8800,-1.8225,79.009375,65.837261,178.5900,-8.4775,88.901637,63.948566,...,0,-0.258819,0.965926,0.433884,-0.900969,3880.0,0.0,565.0,3315.0,23
28701,0.0000,-0.1500,-0.1275,0.0025,90.574479,62.861038,170.7375,-1.1325,81.880074,64.188714,...,1,-0.258819,0.965926,-0.433884,-0.900969,3730.0,0.0,300.0,3430.0,23
28702,-0.0575,-0.1000,-0.5500,-1.0000,88.872917,64.852764,174.6900,-9.0025,75.577917,64.867084,...,0,-0.258819,0.965926,-0.974928,-0.222521,3790.0,0.0,225.0,3565.0,23


In [11]:
prices_train

2022-12-31 22:00:00+00:00    210.0500
2022-12-31 23:00:00+00:00    230.9000
2023-01-01 00:00:00+00:00    268.1900
2023-01-01 01:00:00+00:00    229.5800
2023-01-01 02:00:00+00:00    235.9800
                               ...   
2026-05-12 17:00:00+00:00    151.2075
2026-05-12 18:00:00+00:00    125.5650
2026-05-12 19:00:00+00:00    129.7575
2026-05-12 20:00:00+00:00    122.8300
2026-05-12 21:00:00+00:00    122.8000
Freq: h, Name: price_eur_mwh, Length: 29472, dtype: float64

In [13]:
m_tr

NameError: name 'm_tr' is not defined